# Unsloth로 Qwen2.5-0.5B-Instruct 실행하기

이 실습에서는 4비트로 양자화된 `unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit` 모델을 Google Colab에서 불러와 대화형 추론을 수행합니다.
> **Colab 설정:** 메뉴에서 `런타임 → 런타임 유형 변경 → T4 GPU`를 선택한 뒤 위에서부터 차례대로 실행하세요.
>
> 이 모델은 공개 모델이므로 Hugging Face 토큰이 필요하지 않습니다. 0.5B 모델은 교육과 빠른 실습에 적합하지만, 큰 모델보다 답변의 정확도와 한국어 표현력이 낮을 수 있습니다.

# 라이브러리 설치

Colab의 새 런타임에는 Unsloth가 없으므로 먼저 설치합니다.

In [ ]:
# %%capture
# %pip install -U unsloth

In [ ]:
import unsloth

# GPU와 실행 환경 확인

Unsloth를 다른 Transformers 관련 라이브러리보다 먼저 가져오는 것이 좋습니다.

In [ ]:
from unsloth import FastLanguageModel
import torch

In [ ]:
print(f"PyTorch 버전: {torch.__version__}")
print(f"사용 GPU: {torch.cuda.get_device_name(0)}")

# 4비트 Qwen2.5 모델 로딩

- `max_seq_length`: 한 번에 처리할 수 있는 최대 토큰 길이입니다. 이 실습에서는 메모리와 실행 시간을 고려해 2,048로 제한합니다.
- `dtype=None`: GPU에 맞는 자료형을 Unsloth가 자동 선택합니다.
- `load_in_4bit=True`: 가중치를 4비트로 불러와 GPU 메모리 사용량을 줄입니다.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

In [ ]:
# 학습이 아니라 추론에 맞게 모델을 최적화합니다.
FastLanguageModel.for_inference(model)
model.device

# Qwen 채팅 템플릿 적용

대화형 모델마다 역할을 표시하는 특수 토큰이 다릅니다. 따라서 `<|user|>` 같은 문자열을 직접 붙이지 않고, 해당 토크나이저의 `apply_chat_template()`을 사용해야 합니다.

- `system`: AI의 역할과 행동 방식을 지정합니다.
- `user`: 사용자의 질문입니다.
- `assistant`: 모델이 생성할 답변 역할입니다.
- `add_generation_prompt=True`: 이제 assistant가 답할 차례임을 나타내는 토큰을 덧붙입니다.

In [ ]:
messages = [
    {"role": "system", "content": "너는 중학생에게 과학을 쉽게 설명하는 친절한 AI야."},
    {"role": "user", "content": "지구는 왜 둥글까? 세 문장으로 설명해 줘."},
]

In [ ]:
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

In [ ]:
prompt

In [ ]:
print(prompt)

# 답변 생성

모델의 출력에는 입력 프롬프트 토큰도 함께 들어 있습니다. `input_length` 이후의 토큰만 잘라서 디코딩하면 모델이 새로 생성한 답변만 얻을 수 있습니다.

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)

In [ ]:
inputs

In [ ]:
print(inputs["input_ids"].shape)
print(inputs["attention_mask"].shape)

In [ ]:
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_length=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.05, # 1.0 초과 이미 등장한 토큰의 선택 가능성을 낮춤. (1.0 반복 불이익 없음 — 기본값)
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

In [ ]:
outputs.shape, outputs

In [ ]:
input_length = inputs["input_ids"].shape[-1]
new_tokens = outputs[0, input_length:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
print(response)

# 재사용 가능한 질문·응답 함수

`temperature=0`이면 매번 가장 가능성이 높은 다음 토큰을 고르는 결정적 생성(greedy decoding)을 사용합니다. 0보다 크면 샘플링을 사용하므로 실행할 때마다 답이 조금 달라질 수 있습니다.

In [ ]:
def chat_qa(messages, max_length=1024, temperature=0.7, top_p=0.9):
    """채팅 메시지를 입력받아 Qwen2.5의 답변 문자열을 반환합니다."""
    if not messages:
        raise ValueError("messages에는 최소 한 개의 메시지가 필요합니다.")

    allowed_roles = {"system", "user", "assistant"}
    for message in messages:
        if message.get("role") not in allowed_roles or "content" not in message:
            raise ValueError("각 메시지는 role(system/user/assistant)과 content를 가져야 합니다.")

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    input_length = model_inputs["input_ids"].shape[-1]
    do_sample = temperature > 0

    generation_options = {
        "max_length": max_length,
        "do_sample": do_sample,
        "repetition_penalty": 1.05,
        "eos_token_id": tokenizer.eos_token_id,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if do_sample:
        generation_options.update(
            temperature=temperature,
            top_p=top_p,
        )

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            **generation_options,
        )

    answer_ids = generated_ids[0, input_length:]
    return tokenizer.decode(answer_ids, skip_special_tokens=True).strip()

In [ ]:
reply = chat_qa(
    [
        {"role": "system", "content": "너는 해적 말투로 과학을 설명하는 AI야."},
        {"role": "user", "content": "지구가 둥근 이유를 쉽게 설명해 줘."},
    ],
    temperature=0.7,
    top_p=0.9,
)

print(reply)

### 방법 2: 채팅 메시지를 직접 함수로 변환

Qwen2.5-Instruct는 다음 ChatML 형식을 사용합니다.

```text
<|im_start|>system
시스템 메시지<|im_end|>
<|im_start|>user
사용자 메시지<|im_end|>
<|im_start|>assistant
```

직접 만드는 방식은 특수 토큰의 구조를 이해하는 데 유용하지만, 다른 모델로 바꾸면 반드시 그 모델에 맞게 함수를 수정해야 합니다.

In [ ]:
def make_qwen_prompt(messages, add_generation_prompt=True):
    """메시지를 Qwen2.5 ChatML 프롬프트 문자열로 직접 변환합니다."""

    prompt_parts = []
    for message in messages:
        role = message["role"]
        content = message["content"]
        prompt_parts.append(
            f"<|im_start|>{role}\n{content}<|im_end|>\n"
        )

    if add_generation_prompt:
        prompt_parts.append("<|im_start|>assistant\n")

    return "".join(prompt_parts)

In [ ]:
manual_prompt = make_qwen_prompt(
    messages,
    add_generation_prompt=True,
)
manual_prompt

In [ ]:
print(prompt)